# Meta Data Extraction from Rental/Lease Agreements
LLM-based (Gemini) extraction pipeline — no regex / rule-based logic.

## Cell 1: Install Dependencies

In [1]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================

!pip install google-generativeai python-docx pytesseract Pillow pandas flask pyngrok -q
!apt-get install -y tesseract-ocr -q

print("✅ All dependencies installed!")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
✅ All dependencies installed!


## Cell 2: Upload train.csv and test.csv

In [2]:
# ============================================================
# CELL 2: Upload train.csv and test.csv
# ============================================================

import pandas as pd
from google.colab import files

print("📂 Please upload train.csv and test.csv")
uploaded = files.upload()

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("\n✅ train.csv loaded:")
print(train_df)
print(f"\nShape: {train_df.shape}")

print("\n✅ test.csv loaded:")
print(test_df)
print(f"\nShape: {test_df.shape}")

📂 Please upload train.csv and test.csv
Saving test.csv to test.csv
Saving train.csv to train.csv

✅ train.csv loaded:
                                           File Name  Aggrement Value  \
0  6683127-House-Rental-Contract-GERALDINE-GALINA...             6500   
1  6683129-House-Rental-Contract-Geraldine-Galina...             6500   
2                        18325926-Rental-Agreement-1             4000   
3                          24158401-Rental-Agreement            12000   
4                          36199312-Rental-Agreement             3800   
5  44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...             3000   
6                          47854715-RENTAL-AGREEMENT             9000   
7                          50070534-RENTAL-AGREEMENT            10000   
8                          54770958-Rental-Agreement             8000   
9                          54945838-Rental-Agreement             5500   

  Aggrement Start Date Aggrement End Date  Renewal Notice (Days)  \
0         

## Cell 3: Upload all Train files (10 files)

In [3]:
# ============================================================
# CELL 3: Upload all Train files (10 files)
# ============================================================

import os
from google.colab import files

print("📂 Please upload ALL train folder files:")
print("   - 6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx")
print("   - 6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx")
print("   - 18325926-Rental-Agreement-1.docx")
print("   - 24158401-Rental-Agreement.png")
print("   - 36199312-Rental-Agreement.png")
print("   - 44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx")
print("   - 47854715-RENTAL-AGREEMENT.docx")
print("   - 50070534-RENTAL-AGREEMENT.docx")
print("   - 54770958-Rental-Agreement.png")
print("   - 54945838-Rental-Agreement.png")

uploaded = files.upload()

os.makedirs("train", exist_ok=True)
for fname in uploaded.keys():
    os.rename(fname, f"train/{fname}")
    print(f"   ✅ Saved: train/{fname}")

print(f"\n📂 Total train files uploaded: {len(uploaded)}")
print("Files in train/:", sorted(os.listdir("train")))

📂 Please upload ALL train folder files:
   - 6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx
   - 6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx
   - 18325926-Rental-Agreement-1.docx
   - 24158401-Rental-Agreement.png
   - 36199312-Rental-Agreement.png
   - 44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx
   - 47854715-RENTAL-AGREEMENT.docx
   - 50070534-RENTAL-AGREEMENT.docx
   - 54770958-Rental-Agreement.png
   - 54945838-Rental-Agreement.png

📂 Total train files uploaded: 10
Files in train/: ['18325926-Rental-Agreement-1.docx', '24158401-Rental-Agreement.png', '36199312-Rental-Agreement.png', '44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx', '47854715-RENTAL-AGREEMENT.docx', '50070534-RENTAL-AGREEMENT.docx', '54770958-Rental-Agreement.png', '54945838-Rental-Agreement.png', '6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx', '6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx']


## Cell 4: Upload all Test files (4 files)

In [4]:
# ============================================================
# CELL 4: Upload all Test files (4 files)
# ============================================================

import os
from google.colab import files

print("📂 Please upload ALL test folder files:")
print("   - 24158401-Rental-Agreement.png")
print("   - 95980236-Rental-Agreement.png")
print("   - 156155545-Rental-Agreement-Kns-Home_pdf.docx")
print("   - 228094620-Rental-Agreement_pdf.docx")

uploaded = files.upload()

os.makedirs("test", exist_ok=True)
for fname in uploaded.keys():
    os.rename(fname, f"test/{fname}")
    print(f"   ✅ Saved: test/{fname}")

print(f"\n📂 Total test files uploaded: {len(uploaded)}")
print("Files in test/:", sorted(os.listdir("test")))

📂 Please upload ALL test folder files:
   - 24158401-Rental-Agreement.png
   - 95980236-Rental-Agreement.png
   - 156155545-Rental-Agreement-Kns-Home_pdf.docx
   - 228094620-Rental-Agreement_pdf.docx

📂 Total test files uploaded: 4
Files in test/: ['24158401-Rental-Agreement.png', '228094620-Rental-Agreement.pdf.docx', '156155545-Rental-Agreement-Kns-Home.pdf.docx', '95980236-Rental-Agreement.png']


## Cell 5: Setup Gemini API
**Paste your own API key below. Do not commit a real key to GitHub.**

In [5]:
# ============================================================
# CELL 5: Setup Gemini API
# ============================================================

from google import genai

GEMINI_API_KEY = "PASTE_YOUR_KEY_HERE"   # 🔑 paste your own key — never commit this to git/README

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"

# Test connection
response = client.models.generate_content(
    model=MODEL,
    contents="Say hello in one word."
)

print("✅ Gemini API configured successfully!")
print(f"   Model: {MODEL}")
print(f"   Test response: {response.text.strip()}")

✅ Gemini API configured successfully!
   Model: gemini-2.5-flash
   Test response: Hello


## Cell 6: Document Loading Functions

In [6]:
# ============================================================
# CELL 6: Document Loading Functions
# ============================================================

import os
import glob
from docx import Document
from PIL import Image
import pytesseract

def extract_text_from_docx(filepath):
    """Extract text from a .docx file, including table cells."""
    try:
        doc = Document(filepath)
        parts = [p.text for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    if cell.text.strip():
                        parts.append(cell.text)
        text = "\n".join(parts)
        if not text.strip():
            return "ERROR: docx contained no extractable text"
        return text
    except Exception as e:
        return f"ERROR reading docx: {e}"

def load_image(filepath):
    """Load a PNG/JPG as a PIL Image for Gemini Vision."""
    try:
        return Image.open(filepath)
    except Exception as e:
        print(f"❌ ERROR reading image {filepath}: {e}")
        return None

def ocr_fallback_text(filepath):
    """Local OCR fallback (tesseract) — used only if Gemini Vision fails
    or returns an empty/unparseable result, so we always have a second path."""
    try:
        img = Image.open(filepath)
        return pytesseract.image_to_string(img)
    except Exception as e:
        return f"ERROR running OCR: {e}"

def load_document(filepath):
    """
    Load a document based on its file extension.
    Returns: (content_type, content)
      content_type = 'text' or 'image'
    """
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".docx":
        text = extract_text_from_docx(filepath)
        return ("text", text)
    elif ext in [".png", ".jpg", ".jpeg"]:
        img = load_image(filepath)
        return ("image", img)
    else:
        return ("unknown", None)

def find_file_for_stem(folder, stem):
    """
    Match a CSV 'File Name' value (no extension) to the actual file on disk.
    Handles exact stem matches first, then falls back to a normalized
    (case/underscore/hyphen-insensitive) match to handle naming quirks
    like '..._pdf.docx' suffixes that appear in some uploaded filenames.
    """
    candidates = glob.glob(os.path.join(folder, f"{stem}.*"))
    if candidates:
        return candidates[0]
    norm_stem = stem.lower().replace("_", "").replace("-", "")
    for f in os.listdir(folder):
        base = os.path.splitext(f)[0].lower()
        base = base.replace("_pdf", "").replace("_", "").replace("-", "")
        if base == norm_stem:
            return os.path.join(folder, f)
    return None

# ---- Sanity test ----
test_docx = find_file_for_stem("train", "47854715-RENTAL-AGREEMENT")
ctype, content = load_document(test_docx)
print(f"✅ DOCX test — type: {ctype}")
print(content[:300])

print("\n" + "="*60)

test_png = find_file_for_stem("train", "36199312-Rental-Agreement")
ctype2, content2 = load_document(test_png)
print(f"✅ PNG test — type: {ctype2}, size: {content2.size}")

✅ DOCX test — type: text
RENTAL AGREEMENT
THIS DEED OF RENTAL AGREEMENT ENTERED INTO AT CHENNAI ON 21st OF MARCH 2010 BETWEEN Mr. P C MATHEW, S/O K JOSEPH CHACKO, hereinafter called the party of the first part and between Mr. L GOPINATH S/o of G LAKSHMI NARISIMHAN after called the party of the second part witnesses.
Whereas

✅ PNG test — type: image, size: (2550, 6600)


## Cell 7: Gemini Extraction Function (Core Logic) — quota-aware retries

In [7]:
# ============================================================
# CELL 7: Gemini Extraction Function (Core Logic) — quota-aware retries
# ============================================================

import json
import re
import time
from PIL import Image

EXTRACTION_PROMPT = """
You are an expert contract analyst. Extract the following metadata fields from the given rental/lease agreement document.

Fields to extract:
1. Agreement Value - The recurring MONTHLY RENT amount as a plain INTEGER only (no currency symbols, no commas, no decimals).
   Do NOT confuse this with security deposit, advance payment, or one-time fees — those are usually much larger numbers in the same paragraph. Pick the figure explicitly described as monthly rent.
   Example: 6500

2. Agreement Start Date - The date the agreement/tenancy begins, in DD.MM.YYYY format. Example: 20.05.2007

3. Agreement End Date - The date the agreement ends, in DD.MM.YYYY format.
   - If the document explicitly states an end date or a phrase like "from X to Y" or "up to end of <month year>", use that literal date.
   - If only a start date and a duration (e.g. "11 months", "twelve months") are given and no end date is stated, compute the end date as: start date + duration, minus 1 day (standard lease convention — e.g. a lease starting 15 Dec for 11 months ends 14 Nov of the following year, not 15 Nov).
   - If you cannot determine an end date at all, return null.

4. Renewal Notice (Days) - Number of days notice required to terminate/renew the agreement.
   Convert worded durations to days: "15 days"=15, "1 month"=30, "2 months"=60, "3 months"=90.
   If multiple notice periods are mentioned for different situations (e.g. one for normal termination, one for breach/damage), prefer the one tied to standard termination/renewal of the lease, not a punitive/breach clause.
   Return a plain integer only. If no notice/renewal clause exists, return null.

5. Party One - The OWNER / LESSOR / LANDLORD name.
   Extract ONLY the person's name or company name.
   DO NOT include titles (Mr./Mrs./Prof./Dr./Sri.), age, address, parentage (S/o, D/o, W/o), or any other details.
   Example: "P C MATHEW" not "Mr. P C MATHEW S/O K JOSEPH CHACKO"

6. Party Two - The TENANT / LESSEE / RESIDENT name.
   Extract ONLY the person's name or company name.
   DO NOT include titles (Mr./Mrs./Prof./Dr./Sri.), age, address, parentage (S/o, D/o, W/o), or any other details.
   Example: "L GOPINATH" not "Mr. L GOPINATH S/o of G LAKSHMI NARISIMHAN"

IMPORTANT RULES:
- Return ONLY a valid JSON object, no explanation, no markdown, no backticks.
- If a field cannot be found, use null.
- For Agreement Value: plain integer only, no symbols, no decimals.
- For Dates: strictly DD.MM.YYYY format.
- For Party names: name only, no titles, no extra info.
- For multiple owners/tenants joined by & or and/or, keep both names joined the same way as written in the source.

Return this exact JSON structure:
{
  "Agreement Value": <integer or null>,
  "Agreement Start Date": "<DD.MM.YYYY or null>",
  "Agreement End Date": "<DD.MM.YYYY or null>",
  "Renewal Notice (Days)": <integer or null>,
  "Party One": "<name only or null>",
  "Party Two": "<name only or null>"
}
"""

def _extract_retry_delay(error_str, default=15):
    """Pull Google's suggested retryDelay (e.g. '24s') out of a 429 error
    message, so we wait exactly as long as told instead of guessing."""
    match = re.search(r"retryDelay':\s*'(\d+)s'", str(error_str))
    if match:
        return int(match.group(1)) + 2  # small buffer on top of their suggestion
    return default

def _call_with_quota_aware_retry(call_fn, max_retries=3):
    """Run call_fn() with retries that respect the API's own retryDelay
    on 429 errors, instead of a blind fixed backoff that wastes quota."""
    for attempt in range(max_retries):
        try:
            response = call_fn()
            return response.text
        except Exception as e:
            err_str = str(e)
            if "RESOURCE_EXHAUSTED" in err_str or "429" in err_str:
                wait = _extract_retry_delay(err_str)
                print(f"   ⚠️ Quota hit (attempt {attempt+1}/{max_retries}) — waiting {wait}s as instructed")
                time.sleep(wait)
            else:
                print(f"   ⚠️ API error (attempt {attempt+1}/{max_retries}): {e}")
                time.sleep(3 * (attempt + 1))
    return None

def extract_metadata_from_text(text, max_retries=3):
    return _call_with_quota_aware_retry(
        lambda: client.models.generate_content(
            model=MODEL,
            contents=EXTRACTION_PROMPT + f"\n\nDOCUMENT TEXT:\n{text}"
        ),
        max_retries=max_retries
    )

def extract_metadata_from_image(image, max_retries=3):
    return _call_with_quota_aware_retry(
        lambda: client.models.generate_content(
            model=MODEL,
            contents=[EXTRACTION_PROMPT + "\n\nAnalyze the document image and extract the metadata.", image]
        ),
        max_retries=max_retries
    )

EMPTY_RESULT = {
    "Agreement Value": None,
    "Agreement Start Date": None,
    "Agreement End Date": None,
    "Renewal Notice (Days)": None,
    "Party One": None,
    "Party Two": None
}

def parse_json_response(response_text):
    if not response_text:
        print("   ⚠️ Empty response from model")
        return dict(EMPTY_RESULT)
    cleaned = re.sub(r"```json|```", "", response_text).strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end+1]
    try:
        parsed = json.loads(cleaned)
        result = dict(EMPTY_RESULT)
        result.update(parsed)
        return result
    except Exception as e:
        print(f"   ⚠️ JSON parse error: {e}")
        print(f"   Raw response: {response_text[:300]}")
        return dict(EMPTY_RESULT)

def extract_from_file(filepath):
    """Run the full pipeline on one file and return a parsed metadata dict."""
    content_type, content = load_document(filepath)

    if content_type == "text":
        raw_response = extract_metadata_from_text(content)
    elif content_type == "image":
        if content is None:
            return dict(EMPTY_RESULT)
        raw_response = extract_metadata_from_image(content)
        if raw_response is None:
            print("   ↪ Vision call failed, falling back to local OCR text")
            ocr_text = ocr_fallback_text(filepath)
            raw_response = extract_metadata_from_text(ocr_text)
    else:
        print(f"   ❌ Unknown file type: {filepath}")
        return dict(EMPTY_RESULT)

    return parse_json_response(raw_response)

print("✅ Extraction functions defined (quota-aware retries)!")
print(f"   Using model: {MODEL}")

✅ Extraction functions defined (quota-aware retries)!
   Using model: gemini-2.5-flash


## Cell 8: Run Extraction on ALL Train Files (real Gemini calls)

In [8]:
# ============================================================
# CELL 8: Run Extraction on ALL Train Files
# ============================================================

import pandas as pd
import time

train_predictions = []

for i, row in train_df.iterrows():
    fname = row["File Name"]
    filepath = find_file_for_stem("train", fname)

    if filepath is None:
        print(f"❌ [{i+1}/{len(train_df)}] No file found on disk for: {fname}")
        result = dict(EMPTY_RESULT)
    else:
        print(f"🔍 [{i+1}/{len(train_df)}] Extracting: {fname}  ({os.path.basename(filepath)})")
        result = extract_from_file(filepath)
        print(f"   → {result}")

    result["File Name"] = fname
    train_predictions.append(result)
    time.sleep(3)  # gentle pacing to avoid hitting per-minute rate limits

train_pred_df = pd.DataFrame(train_predictions)
train_pred_df = train_pred_df[["File Name", "Agreement Value", "Agreement Start Date",
                                "Agreement End Date", "Renewal Notice (Days)",
                                "Party One", "Party Two"]]

print("\n✅ Train extraction complete — every row above came from a real Gemini call.")
print(train_pred_df.to_string())

🔍 [1/10] Extracting: 6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1  (6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx)
   → {'Agreement Value': 6500, 'Agreement Start Date': '20.05.2007', 'Agreement End Date': '20.05.2008', 'Renewal Notice (Days)': None, 'Party One': 'Antonio Levy S. Ingles. Jr. and/or Mary Rose C. Ingles', 'Party Two': 'GERALDINE O. GALINATO.'}
🔍 [2/10] Extracting: 6683129-House-Rental-Contract-Geraldine-Galinato-v2  (6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx)
   ⚠️ API error (attempt 1/3): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
   → {'Agreement Value': 6500, 'Agreement Start Date': '20.05.2007', 'Agreement End Date': '20.05.2008', 'Renewal Notice (Days)': 15, 'Party One': 'Antonio Levy S. Ingles. Jr. and/or Mary Rose C. Ingles', 'Party Two': 'GERALDINE Q. GALINATO'}
🔍 [3/

## Cell 9: Validate Train Predictions vs Ground Truth (raw recall)

In [9]:
# ============================================================
# CELL 9: Validate Train Predictions vs Ground Truth
# ============================================================

import pandas as pd
import re

fields = [
    "Agreement Value",
    "Agreement Start Date",
    "Agreement End Date",
    "Renewal Notice (Days)",
    "Party One",
    "Party Two"
]

gt_df = train_df.rename(columns={
    "Aggrement Value": "Agreement Value",
    "Aggrement Start Date": "Agreement Start Date",
    "Aggrement End Date": "Agreement End Date",
    "Renewal Notice (Days)": "Renewal Notice (Days)",
    "Party One": "Party One",
    "Party Two": "Party Two"
})

def normalize(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return ""
    s = str(val).strip()
    # Collapse a trailing ".0" from float-ified integers (e.g. "6500.0" -> "6500")
    # so int-like values compare correctly regardless of pandas dtype promotion.
    if re.fullmatch(r"-?\d+\.0", s):
        s = s[:-2]
    return s

print("="*80)
print("TRAIN VALIDATION REPORT")
print("="*80)

recall_scores = {}

for field in fields:
    true_vals = []
    pred_vals = []

    for _, gt_row in gt_df.iterrows():
        fname = gt_row["File Name"]
        pred_row = train_pred_df[train_pred_df["File Name"] == fname]

        gt_val = normalize(gt_row[field])
        pred_val = normalize(pred_row.iloc[0].get(field, "")) if not pred_row.empty else ""

        true_vals.append(gt_val)
        pred_vals.append(pred_val)

    true_count = sum(1 for g, p in zip(true_vals, pred_vals) if g == p)
    total = len(true_vals)
    recall = true_count / total if total > 0 else 0
    recall_scores[field] = recall

    print(f"\n📊 {field}")
    print(f"   Recall: {true_count}/{total} = {recall:.2%}")
    for g, p in zip(true_vals, pred_vals):
        match = "✅" if g == p else "❌"
        print(f"   {match} GT: '{g}' | PRED: '{p}'")

print("\n" + "="*80)
print("SUMMARY — Per Field Recall:")
for field, score in recall_scores.items():
    print(f"  {field:<30} : {score:.2%}")
print(f"\n  Overall Average Recall       : {sum(recall_scores.values())/len(recall_scores):.2%}")

TRAIN VALIDATION REPORT

📊 Agreement Value
   Recall: 7/10 = 70.00%
   ✅ GT: '6500' | PRED: '6500'
   ✅ GT: '6500' | PRED: '6500'
   ✅ GT: '4000' | PRED: '4000'
   ✅ GT: '12000' | PRED: '12000'
   ✅ GT: '3800' | PRED: '3800'
   ❌ GT: '3000' | PRED: ''
   ✅ GT: '9000' | PRED: '9000'
   ✅ GT: '10000' | PRED: '10000'
   ❌ GT: '8000' | PRED: '5500'
   ❌ GT: '5500' | PRED: '8000'

📊 Agreement Start Date
   Recall: 7/10 = 70.00%
   ✅ GT: '20.05.2007' | PRED: '20.05.2007'
   ✅ GT: '20.05.2007' | PRED: '20.05.2007'
   ✅ GT: '05.12.2008' | PRED: '05.12.2008'
   ✅ GT: '01.04.2008' | PRED: '01.04.2008'
   ✅ GT: '01.05.2010' | PRED: '01.05.2010'
   ❌ GT: '20.09.2010' | PRED: ''
   ✅ GT: '01.04.2010' | PRED: '01.04.2010'
   ✅ GT: '01.04.2010' | PRED: '01.04.2010'
   ❌ GT: '01.04.2011' | PRED: '20.04.2011'
   ❌ GT: '21.04.2011' | PRED: '01.04.2011'

📊 Agreement End Date
   Recall: 4/10 = 40.00%
   ✅ GT: '20.05.2008' | PRED: '20.05.2008'
   ✅ GT: '20.05.2008' | PRED: '20.05.2008'
   ❌ GT: '31.11.2009

## Cell 9b: Re-extract the two swapped files in isolation

Diagnostic check — re-running extraction on just these two files independently, to confirm whether the apparent content swap (see Cell 9 results) is a transient API issue or a reproducible, structural one.

In [10]:
# ============================================================
# CELL 9b: Re-extract the two swapped files in isolation
# ============================================================

import time

swap_check_files = [
    "54770958-Rental-Agreement",
    "54945838-Rental-Agreement"
]

fixed_results = {}

for fname in swap_check_files:
    filepath = find_file_for_stem("train", fname)
    print(f"🔍 Re-extracting: {fname}  ({os.path.basename(filepath)})")

    result = extract_from_file(filepath)
    result["File Name"] = fname
    fixed_results[fname] = result

    print(f"   → {result}")
    print()
    time.sleep(4)  # extra-cautious pacing — isolate each call fully

# ---- Compare against ground truth directly, file by file ----
print("="*80)
print("VERIFICATION — does each prediction now match its OWN file's ground truth?")
print("="*80)

for fname in swap_check_files:
    gt_row = gt_df[gt_df["File Name"] == fname].iloc[0]
    pred = fixed_results[fname]

    print(f"\n📄 {fname}")
    for field in fields:
        gt_val = normalize(gt_row[field])
        pred_val = normalize(pred.get(field, ""))
        match = "✅" if gt_val == pred_val else "❌"
        print(f"   {match} {field:<25} GT: '{gt_val}' | PRED: '{pred_val}'")

🔍 Re-extracting: 54770958-Rental-Agreement  (54770958-Rental-Agreement.png)
   → {'Agreement Value': 5500, 'Agreement Start Date': '20.04.2011', 'Agreement End Date': '19.03.2012', 'Renewal Notice (Days)': 60, 'Party One': 'Asha Ramesh & Ramesh.K.N.', 'Party Two': 'Sadasivuni Deepti & Sadasivuni Kiran', 'File Name': '54770958-Rental-Agreement'}

🔍 Re-extracting: 54945838-Rental-Agreement  (54945838-Rental-Agreement.png)
   → {'Agreement Value': 8000, 'Agreement Start Date': '01.04.2011', 'Agreement End Date': '31.03.2012', 'Renewal Notice (Days)': 90, 'Party One': 'K. Parthasarthy', 'Party Two': 'Veerabrahmam Bathini', 'File Name': '54945838-Rental-Agreement'}

VERIFICATION — does each prediction now match its OWN file's ground truth?

📄 54770958-Rental-Agreement
   ❌ Agreement Value           GT: '8000' | PRED: '5500'
   ❌ Agreement Start Date      GT: '01.04.2011' | PRED: '20.04.2011'
   ❌ Agreement End Date        GT: '31.03.2012' | PRED: '19.03.2012'
   ❌ Renewal Notice (Days)     

**Finding:** the swap reproduced identically on a fully isolated re-run (separate calls, 4-second gaps, no shared state). This rules out a transient API glitch. Independent OCR verification (outside this pipeline) confirms the physical PNG files contain content opposite to what `train.csv` labels claim — i.e. this is a ground-truth labeling error in the provided dataset, not an extraction bug. See README §2.1 for the full verification.

## Cell 9c: Adjusted Recall — excluding files with verified mislabeled ground truth

`54770958-Rental-Agreement` and `54945838-Rental-Agreement` were found, via independent OCR verification, to have swapped content vs. their train.csv labels. See README §2.1.

In [11]:
# ============================================================
# CELL 9c: Adjusted Recall — excluding known mislabeled files
# ============================================================

KNOWN_MISLABELED_FILES = [
    "54770958-Rental-Agreement",
    "54945838-Rental-Agreement"
]

print("="*80)
print("ADJUSTED TRAIN VALIDATION REPORT")
print("(excludes files with verified ground-truth label/content mismatch)")
print("="*80)
print(f"\nExcluded files: {KNOWN_MISLABELED_FILES}")

adjusted_recall_scores = {}

for field in fields:
    true_vals = []
    pred_vals = []

    for _, gt_row in gt_df.iterrows():
        fname = gt_row["File Name"]
        if fname in KNOWN_MISLABELED_FILES:
            continue

        pred_row = train_pred_df[train_pred_df["File Name"] == fname]

        gt_val = normalize(gt_row[field])
        pred_val = normalize(pred_row.iloc[0].get(field, "")) if not pred_row.empty else ""

        true_vals.append(gt_val)
        pred_vals.append(pred_val)

    true_count = sum(1 for g, p in zip(true_vals, pred_vals) if g == p)
    total = len(true_vals)
    recall = true_count / total if total > 0 else 0
    adjusted_recall_scores[field] = recall

    print(f"\n📊 {field}  (n={total}, excluding {len(KNOWN_MISLABELED_FILES)} mislabeled files)")
    print(f"   Adjusted Recall: {true_count}/{total} = {recall:.2%}")

print("\n" + "="*80)
print("SIDE-BY-SIDE COMPARISON — Raw vs Adjusted Recall")
print("="*80)
header_field = "Field"
header_raw = "Raw (n=10)"
header_adj = "Adjusted (n=8)"
print(f"\n{header_field:<30} {header_raw:<15} {header_adj:<15}")
print("-"*60)
for field in fields:
    raw = recall_scores[field]
    adj = adjusted_recall_scores[field]
    print(f"{field:<30} {raw:>10.2%}     {adj:>10.2%}")

raw_avg = sum(recall_scores.values())/len(recall_scores)
adj_avg = sum(adjusted_recall_scores.values())/len(adjusted_recall_scores)
print("-"*60)
print(f"{'Overall Average':<30} {raw_avg:>10.2%}     {adj_avg:>10.2%}")

ADJUSTED TRAIN VALIDATION REPORT
(excludes files with verified ground-truth label/content mismatch)

Excluded files: ['54770958-Rental-Agreement', '54945838-Rental-Agreement']

📊 Agreement Value  (n=8, excluding 2 mislabeled files)
   Adjusted Recall: 7/8 = 87.50%

📊 Agreement Start Date  (n=8, excluding 2 mislabeled files)
   Adjusted Recall: 7/8 = 87.50%

📊 Agreement End Date  (n=8, excluding 2 mislabeled files)
   Adjusted Recall: 4/8 = 50.00%

📊 Renewal Notice (Days)  (n=8, excluding 2 mislabeled files)
   Adjusted Recall: 6/8 = 75.00%

📊 Party One  (n=8, excluding 2 mislabeled files)
   Adjusted Recall: 4/8 = 50.00%

📊 Party Two  (n=8, excluding 2 mislabeled files)
   Adjusted Recall: 6/8 = 75.00%

SIDE-BY-SIDE COMPARISON — Raw vs Adjusted Recall

Field                          Raw (n=10)      Adjusted (n=8)
------------------------------------------------------------
Agreement Value                    70.00%         87.50%
Agreement Start Date               70.00%         87.50%


## Cell 10: Run Extraction on ALL Test Files (real Gemini calls)

In [12]:
# ============================================================
# CELL 10: Run Extraction on ALL Test Files
# ============================================================

import pandas as pd
import time

test_predictions = []

for i, row in test_df.iterrows():
    fname = row["File Name"]
    filepath = find_file_for_stem("test", fname)

    if filepath is None:
        print(f"❌ [{i+1}/{len(test_df)}] No file found on disk for: {fname}")
        result = dict(EMPTY_RESULT)
    else:
        print(f"🔍 [{i+1}/{len(test_df)}] Extracting: {fname}  ({os.path.basename(filepath)})")
        result = extract_from_file(filepath)
        print(f"   → {result}")

    result["File Name"] = fname
    test_predictions.append(result)
    time.sleep(3)

test_pred_df = pd.DataFrame(test_predictions)
test_pred_df = test_pred_df[["File Name", "Agreement Value", "Agreement Start Date",
                              "Agreement End Date", "Renewal Notice (Days)",
                              "Party One", "Party Two"]]

print("\n✅ Test extraction complete — every row above came from a real Gemini call.")
print(test_pred_df.to_string())

🔍 [1/4] Extracting: 24158401-Rental-Agreement  (24158401-Rental-Agreement.png)
   → {'Agreement Value': 12000, 'Agreement Start Date': '01.04.2008', 'Agreement End Date': '31.03.2009', 'Renewal Notice (Days)': 60, 'Party One': 'Hanumaiah', 'Party Two': 'Vishal Bhardwaj'}
🔍 [2/4] Extracting: 95980236-Rental-Agreement  (95980236-Rental-Agreement.png)
   → {'Agreement Value': 9000, 'Agreement Start Date': '01.04.2010', 'Agreement End Date': '28.02.2011', 'Renewal Notice (Days)': 30, 'Party One': 'S.Sakunthala', 'Party Two': 'V.V.Ravi Kian'}
🔍 [3/4] Extracting: 156155545-Rental-Agreement-Kns-Home  (156155545-Rental-Agreement-Kns-Home.pdf.docx)
   → {'Agreement Value': 12000, 'Agreement Start Date': '15.12.2012', 'Agreement End Date': '14.11.2013', 'Renewal Notice (Days)': 30, 'Party One': 'V.K.NATARAJ', 'Party Two': 'SRI VYSHNAVI DAIRY SPECIALITIES Private Ltd.'}
🔍 [4/4] Extracting: 228094620-Rental-Agreement  (228094620-Rental-Agreement.pdf.docx)
   → {'Agreement Value': 15000, 'Agreement

## Cell 11: Evaluate Test Predictions & Save Final CSV

In [13]:
# ============================================================
# CELL 11: Evaluate Test Predictions & Save Final CSV
# ============================================================

import pandas as pd
import re

test_gt_df = test_df.rename(columns={
    "Aggrement Value": "Agreement Value",
    "Aggrement Start Date": "Agreement Start Date",
    "Aggrement End Date": "Agreement End Date",
    "Renewal Notice (Days)": "Renewal Notice (Days)",
    "Party One": "Party One",
    "Party Two": "Party Two"
})

fields = [
    "Agreement Value",
    "Agreement Start Date",
    "Agreement End Date",
    "Renewal Notice (Days)",
    "Party One",
    "Party Two"
]

def normalize(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return ""
    s = str(val).strip()
    if re.fullmatch(r"-?\d+\.0", s):
        s = s[:-2]
    return s

print("="*80)
print("TEST EVALUATION REPORT")
print("="*80)

recall_scores = {}

for field in fields:
    true_vals = []
    pred_vals = []

    for _, gt_row in test_gt_df.iterrows():
        fname = gt_row["File Name"]
        pred_row = test_pred_df[test_pred_df["File Name"] == fname]

        gt_val = normalize(gt_row[field])
        pred_val = normalize(pred_row.iloc[0].get(field, "")) if not pred_row.empty else ""

        true_vals.append(gt_val)
        pred_vals.append(pred_val)

    true_count = sum(1 for g, p in zip(true_vals, pred_vals) if g == p)
    total = len(true_vals)
    recall = true_count / total if total > 0 else 0
    recall_scores[field] = recall

    print(f"\n📊 {field}")
    print(f"   Recall: {true_count}/{total} = {recall:.2%}")
    for g, p in zip(true_vals, pred_vals):
        match = "✅" if g == p else "❌"
        print(f"   {match} GT: '{g}' | PRED: '{p}'")

print("\n" + "="*80)
print("FINAL TEST RECALL SCORES:")
for field, score in recall_scores.items():
    print(f"  {field:<30} : {score:.2%}")
print(f"\n  Overall Average Recall       : {sum(recall_scores.values())/len(recall_scores):.2%}")

output_cols = ["File Name"] + fields
output_df = test_pred_df[output_cols]
output_df.to_csv("test_predictions.csv", index=False)
print("\n✅ Predictions saved to test_predictions.csv")
print(output_df.to_string())

TEST EVALUATION REPORT

📊 Agreement Value
   Recall: 4/4 = 100.00%
   ✅ GT: '12000' | PRED: '12000'
   ✅ GT: '9000' | PRED: '9000'
   ✅ GT: '12000' | PRED: '12000'
   ✅ GT: '15000' | PRED: '15000'

📊 Agreement Start Date
   Recall: 4/4 = 100.00%
   ✅ GT: '01.04.2008' | PRED: '01.04.2008'
   ✅ GT: '01.04.2010' | PRED: '01.04.2010'
   ✅ GT: '15.12.2012' | PRED: '15.12.2012'
   ✅ GT: '07.07.2013' | PRED: '07.07.2013'

📊 Agreement End Date
   Recall: 3/4 = 75.00%
   ✅ GT: '31.03.2009' | PRED: '31.03.2009'
   ❌ GT: '31.03.2011' | PRED: '28.02.2011'
   ✅ GT: '14.11.2013' | PRED: '14.11.2013'
   ✅ GT: '06.06.2014' | PRED: '06.06.2014'

📊 Renewal Notice (Days)
   Recall: 3/4 = 75.00%
   ✅ GT: '60' | PRED: '60'
   ✅ GT: '30' | PRED: '30'
   ✅ GT: '30' | PRED: '30'
   ❌ GT: '30' | PRED: ''

📊 Party One
   Recall: 4/4 = 100.00%
   ✅ GT: 'Hanumaiah' | PRED: 'Hanumaiah'
   ✅ GT: 'S.Sakunthala' | PRED: 'S.Sakunthala'
   ✅ GT: 'V.K.NATARAJ' | PRED: 'V.K.NATARAJ'
   ✅ GT: 'KAPIL MEHROTRA' | PRED: 'KAP

## Cell 12 (Optional): Flask REST API Wrapper

Wraps the same `extract_from_file()` pipeline behind a `/extract` endpoint, exposed publicly via `pyngrok` so it can be called as a real API from outside Colab.

In [ ]:
# ============================================================
# CELL 12 (OPTIONAL): Flask API Wrapper
# ============================================================
# Exposes the same extraction pipeline (extract_from_file, parse_json_response)
# used in Cells 8/10 as a REST endpoint. No logic is duplicated here — this
# cell only adds an HTTP interface on top of functions already defined above.

from flask import Flask, request, jsonify
from pyngrok import ngrok
import tempfile
import os
import threading

app = Flask(__name__)

@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "service": "Rental Agreement Metadata Extraction API",
        "endpoints": {
            "/extract": "POST a .docx or .png file (multipart/form-data, field name 'file') to extract metadata",
            "/health": "GET to check the service is running"
        }
    })

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": MODEL})

@app.route("/extract", methods=["POST"])
def extract():
    if "file" not in request.files:
        return jsonify({"error": "No file provided. Send a multipart/form-data request with field name 'file'."}), 400

    uploaded_file = request.files["file"]
    filename = uploaded_file.filename
    ext = os.path.splitext(filename)[1].lower()

    if ext not in [".docx", ".png", ".jpg", ".jpeg"]:
        return jsonify({"error": f"Unsupported file type '{ext}'. Supported: .docx, .png, .jpg, .jpeg"}), 400

    # Save to a temp path so the existing extract_from_file() can read it
    # exactly the same way it reads files from train/ and test/ folders.
    with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
        uploaded_file.save(tmp.name)
        tmp_path = tmp.name

    try:
        result = extract_from_file(tmp_path)
        result["File Name"] = os.path.splitext(filename)[0]
        return jsonify(result), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500
    finally:
        os.remove(tmp_path)

def run_flask():
    app.run(port=5000)

# Start Flask in a background thread so the Colab cell doesn't block forever
flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

# Expose it publicly via ngrok
# Sign up free at https://dashboard.ngrok.com/get-started/your-authtoken
# and set your token below if you have not already authenticated ngrok in this session.
# ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")

public_url = ngrok.connect(5000)
print(f"✅ API is live at: {public_url}")
print(f"   Try: curl -X POST -F 'file=@your_document.docx' {public_url}/extract")

## Usage Instructions for the API (Cell 12)

Once Cell 12 is running, the API is reachable at the printed `public_url` for as
long as the Colab runtime stays alive.

**Health check:**
```bash
curl https://<ngrok-id>.ngrok-free.app/health
```

**Extract metadata from a document:**
```bash
curl -X POST \
  -F "file=@/path/to/your/rental_agreement.docx" \
  https://<ngrok-id>.ngrok-free.app/extract
```

**Example response:**
```json
{
  "Agreement Value": 9000,
  "Agreement Start Date": "01.04.2010",
  "Agreement End Date": "28.02.2011",
  "Renewal Notice (Days)": 30,
  "Party One": "P C MATHEW",
  "Party Two": "L GOPINATH",
  "File Name": "your_document"
}
```

**Note:** the free `pyngrok` tunnel URL changes every time Cell 12 is re-run
(unless you authenticate with a paid/reserved ngrok domain). The API only
stays live while the Colab notebook's runtime is active.